# 🫁 Pneumonia Detection from Chest X-Rays
### Deep Learning with DenseNet121 Transfer Learning

---
**Pipeline:**
1. Exploratory Data Analysis (EDA)
2. Data Loading & Augmentation
3. Model Architecture
4. Training
5. Evaluation (Accuracy, AUC-ROC, Sensitivity, Specificity)
6. Grad-CAM Visualization
7. Single Image Prediction

---
> **Dataset:** [Chest X-Ray Images (Pneumonia) — Kaggle](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia)

In [ ]:
# ─── Setup: add project root to path ───────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image
import torch

import config

print(f'PyTorch version : {torch.__version__}')
print(f'Device          : {config.DEVICE}')
print(f'CUDA available  : {torch.cuda.is_available()}')

## 1. Exploratory Data Analysis

In [ ]:
# ─── Count images per split ────────────────────────────────────────────────
splits = {'train': config.TRAIN_DIR,
          'val'  : config.VAL_DIR,
          'test' : config.TEST_DIR}

records = []
for split, path in splits.items():
    for cls in config.CLASSES:
        cls_path = os.path.join(path, cls)
        if os.path.isdir(cls_path):
            n = len([f for f in os.listdir(cls_path)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        else:
            n = 0
        records.append({'split': split, 'class': cls, 'count': n})

df = pd.DataFrame(records)
print(df.pivot(index='class', columns='split', values='count').fillna(0).astype(int))

In [ ]:
# ─── Class distribution bar chart ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

colors = {'NORMAL': '#00b894', 'PNEUMONIA': '#d63031'}

for ax, split in zip(axes, ['train', 'val', 'test']):
    sub = df[df['split'] == split]
    bars = ax.bar(sub['class'], sub['count'],
                  color=[colors[c] for c in sub['class']])
    ax.set_title(f'{split.capitalize()} Set', fontsize=13, fontweight='bold')
    ax.set_ylabel('Number of Images')
    ax.set_ylim(0, sub['count'].max() * 1.2)
    for bar, row in zip(bars, sub.itertuples()):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 20,
                str(row.count), ha='center', fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Class Distribution per Split', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(config.RESULTS_DIR, 'class_distribution.png'),
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Sample images grid ─────────────────────────────────────────────────────
def show_sample_grid(n_per_class=4):
    fig, axes = plt.subplots(2, n_per_class, figsize=(4 * n_per_class, 8))
    fig.suptitle('Sample Chest X-Rays', fontsize=16, fontweight='bold')

    for row, cls in enumerate(config.CLASSES):
        cls_dir = os.path.join(config.TRAIN_DIR, cls)
        files   = [f for f in os.listdir(cls_dir)
                   if f.lower().endswith(('.jpg', '.jpeg', '.png'))][:n_per_class]
        for col, fname in enumerate(files):
            img = Image.open(os.path.join(cls_dir, fname)).convert('RGB')
            axes[row, col].imshow(img, cmap='gray')
            axes[row, col].set_title(cls, color=colors[cls], fontweight='bold')
            axes[row, col].axis('off')

    plt.tight_layout()
    plt.savefig(os.path.join(config.RESULTS_DIR, 'sample_images.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

show_sample_grid()

In [ ]:
# ─── Image size distribution ────────────────────────────────────────────────
widths, heights = [], []
sample_dir = os.path.join(config.TRAIN_DIR, 'PNEUMONIA')
files = [f for f in os.listdir(sample_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))]

for fname in files[:300]:   # sample 300 images
    img = Image.open(os.path.join(sample_dir, fname))
    w, h = img.size
    widths.append(w)
    heights.append(h)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths,  bins=30, color='#6c5ce7', edgecolor='white')
axes[0].set_title('Image Width Distribution')
axes[0].set_xlabel('Width (px)')

axes[1].hist(heights, bins=30, color='#fd79a8', edgecolor='white')
axes[1].set_title('Image Height Distribution')
axes[1].set_xlabel('Height (px)')

for ax in axes:
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Raw Image Size Distribution (sample: 300)', fontsize=13)
plt.tight_layout()
plt.show()
print(f'Width  — mean: {np.mean(widths):.0f}px  std: {np.std(widths):.0f}px')
print(f'Height — mean: {np.mean(heights):.0f}px  std: {np.std(heights):.0f}px')

## 2. Data Augmentation Preview

In [ ]:
# ─── Show augmented versions of the same image ─────────────────────────────
from dataset import get_transforms, ChestXRayDataset

train_tf = get_transforms('train')
test_tf  = get_transforms('test')

# Pick any PNEUMONIA image
pneu_dir = os.path.join(config.TRAIN_DIR, 'PNEUMONIA')
sample_img_path = os.path.join(pneu_dir,
    [f for f in os.listdir(pneu_dir) if f.endswith('.jpeg')][0])

pil_img = Image.open(sample_img_path).convert('RGB')

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
fig.suptitle('Data Augmentation Examples (PNEUMONIA image)', fontsize=14, fontweight='bold')

# Row 0: original
for col in range(5):
    t = test_tf(pil_img).permute(1, 2, 0).numpy()
    t = (t * np.array(config.STD) + np.array(config.MEAN)).clip(0, 1)
    axes[0, col].imshow(t)
    axes[0, col].set_title('Original (no aug)', fontsize=9)
    axes[0, col].axis('off')

# Row 1: augmented
for col in range(5):
    t = train_tf(pil_img).permute(1, 2, 0).numpy()
    t = (t * np.array(config.STD) + np.array(config.MEAN)).clip(0, 1)
    axes[1, col].imshow(t)
    axes[1, col].set_title('Augmented', fontsize=9)
    axes[1, col].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(config.RESULTS_DIR, 'augmentation_examples.png'),
            dpi=150, bbox_inches='tight')
plt.show()

## 3. Model Architecture

In [ ]:
from model import get_model
from utils import count_parameters, print_device_info

print_device_info()
model = get_model(pretrained=True)
print(f'\nTrainable params: {count_parameters(model):,}')

# Show last few layers
print('\nClassifier head:')
if hasattr(model, 'classifier'):
    print(model.classifier)
elif hasattr(model, 'fc'):
    print(model.fc)

## 4. Training

In [ ]:
# ─── Run training ───────────────────────────────────────────────────────────
# NOTE: This will train the full model. May take 20-60 min depending on GPU.
# Set NUM_EPOCHS in config.py to a smaller number (e.g. 3) for a quick test.

from train import train
model, tracker = train()

## 5. Evaluation

In [ ]:
from evaluate import evaluate
metrics = evaluate(model=model)
print('\nFinal metrics:')
for k, v in metrics.items():
    print(f'  {k:15s}: {v:.2f}%')

## 6. Grad-CAM Visualization

In [ ]:
from gradcam import run_gradcam

# Visualize a PNEUMONIA case
pneu_dir = os.path.join(config.TEST_DIR, 'PNEUMONIA')
pneu_img = os.path.join(pneu_dir,
    [f for f in os.listdir(pneu_dir) if f.lower().endswith(('.jpeg','.jpg'))][0])

print('Grad-CAM for PNEUMONIA image:')
run_gradcam(pneu_img, save_dir=config.RESULTS_DIR)

In [ ]:
# Visualize a NORMAL case
norm_dir = os.path.join(config.TEST_DIR, 'NORMAL')
norm_img = os.path.join(norm_dir,
    [f for f in os.listdir(norm_dir) if f.lower().endswith(('.jpeg','.jpg'))][0])

print('Grad-CAM for NORMAL image:')
run_gradcam(norm_img, save_dir=config.RESULTS_DIR)

## 7. Single Image Prediction

In [ ]:
from predict import predict_image

# Test on a pneumonia X-ray
result = predict_image(
    image_path = pneu_img,
    model      = model,
    show_plot  = True,
    save_path  = os.path.join(config.RESULTS_DIR, 'prediction_example.png')
)

import json
print(json.dumps(result, indent=2))

## Summary

| Component | Details |
|-----------|--------|
| **Dataset** | Chest X-Ray Images (Pneumonia) — Kaggle |
| **Architecture** | DenseNet121 pretrained on ImageNet |
| **Input size** | 224 × 224 RGB |
| **Loss** | Weighted CrossEntropy (handles class imbalance) |
| **Optimizer** | Adam (lr=1e-4, weight_decay=1e-4) |
| **Scheduler** | StepLR (step=7, gamma=0.1) |
| **Augmentation** | Flip, Rotation, ColorJitter, Affine |
| **Explainability** | Grad-CAM on last conv block |

### Clinical Consideration
> **Sensitivity** (recall for pneumonia) is prioritized over specificity because a **false negative** (missing pneumonia) is far more dangerous clinically than a **false positive** (unnecessary follow-up).